In [5]:
#packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.pyplot import subplots
import statsmodels.api as sm
from ISLP import (load_data, confusion_table)
from ISLP.models import (ModelSpec as MS, summarize, contrast)
from sklearn.model_selection import train_test_split 
# from sklearn.naive_bayes import GaussianNB
# from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import RocCurveDisplay, roc_auc_score

roc_curve_est = RocCurveDisplay.from_estimator 
roc_curve_pred = RocCurveDisplay.from_predictions 

### This project will use the [Sleep, Health, and Lifestyle dataset](https://www.kaggle.com/datasets/ayeshaimran1619/sleep-and-lifestyle-study) from Kaggle.

In [6]:
sleep = pd.read_csv('data/Sleep_health_and_lifestyle_dataset.csv')
sleep.head()

,Person ID,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Blood Pressure,Heart Rate,Daily Steps,Sleep Disorder
0,1,Male,27,Software Engineer,6.1,6,42,6,Overweight,126/83,77,4200,NaN
1,2,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,NaN
2,3,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,NaN
3,4,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea
4,5,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea


### Data Preprocessing

#### - Split blood pressure into systolic (artery pressure during contractions) and diastolic (pressure when heart relaxes between beats)
#### - Use one-hot encoding for categorical variables

In [3]:
sleep[['Systolic BP', 'Diastolic BP']] = sleep['Blood Pressure'].str.split('/', expand = True).astype(int)

sleep['BMI Category'] = sleep['BMI Category'].replace('Normal Weight', 'Normal')

sleep['Has Disorder'] = sleep['Sleep Disorder'].notna().astype(int)

sleep = sleep.drop(['Blood Pressure', 'Sleep Disorder', 'Person ID'], axis=1)

categorical_columns = ['Gender', 'Occupation', 'BMI Category']
sleep = pd.get_dummies(sleep, columns = categorical_columns, drop_first = True, dtype = int)

sleep

,Age,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,Heart Rate,Daily Steps,Systolic BP,Diastolic BP,Has Disorder,...,Occupation_Lawyer,Occupation_Manager,Occupation_Nurse,Occupation_Sales Representative,Occupation_Salesperson,Occupation_Scientist,Occupation_Software Engineer,Occupation_Teacher,BMI Category_Obese,BMI Category_Overweight
0,27,6.1,6,42,6,77,4200,126,83,0,...,0,0,0,0,0,0,1,0,0,1
1,28,6.2,6,60,8,75,10000,125,80,0,...,0,0,0,0,0,0,0,0,0,0
2,28,6.2,6,60,8,75,10000,125,80,0,...,0,0,0,0,0,0,0,0,0,0
3,28,5.9,4,30,8,85,3000,140,90,1,...,0,0,0,1,0,0,0,0,1,0
4,28,5.9,4,30,8,85,3000,140,90,1,...,0,0,0,1,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
369,59,8.1,9,75,3,68,7000,140,95,1,...,0,0,1,0,0,0,0,0,0,1
370,59,8.0,9,75,3,68,7000,140,95,1,...,0,0,1,0,0,0,0,0,0,1
371,59,8.1,9,75,3,68,7000,140,95,1,...,0,0,1,0,0,0,0,0,0,1
372,59,8.1,9,75,3,68,7000,140,95,1,...,0,0,1,0,0,0,0,0,0,1


### To begin, we will use Logistic Regression to determine the presence of a sleep disorder rather than a specific diagnosis.

In [5]:
# isolate binary sleep disorders column (target variable)
X = sleep.drop('Has Disorder', axis=1)
y = sleep['Has Disorder']

x_train, x_test, y_train, y_test = train_test_split(X, y,
                                                    random_state=314159,
                                                    test_size=0.33,
                                                    shuffle=True)

# add dummy variables 
x_train['intercept'] = np.ones(x_train.shape[0])
x_test['intercept'] = np.ones(x_test.shape[0])

### Scale features using min-max regularization

In [6]:
needs_scaled = ['Age', 'Sleep Duration', 'Quality of Sleep', 'Physical Activity Level', 'Stress Level', 'Heart Rate', 'Daily Steps', 'Systolic BP', 'Diastolic BP']

for col in needs_scaled:
    col_min = x_train[col].min()
    col_max = x_train[col].max()

    x_train[col] = (x_train[col] - col_min) / (col_max - col_min)
    
    x_test[col] = (x_test[col] - col_min) / (col_max - col_min)

In [7]:
initial_glm = sm.GLM(y_train,
             x_train,
             family=sm.families.Binomial())

# fit model
initial_results = initial_glm.fit()

# analyze model
summarize(initial_results)

,coef,std err,z,P>|z|
Age,3.828600e+00,6.592000e+00,0.581,0.561
Sleep Duration,4.850000e-02,5.357000e+00,0.009,0.993
Quality of Sleep,-1.326800e+01,1.022800e+01,-1.297,0.195
Physical Activity Level,-3.107100e+00,2.765000e+00,-1.124,0.261
Stress Level,-4.905300e+00,6.147000e+00,-0.798,0.425
Heart Rate,-1.540500e+00,9.693000e+00,-0.159,0.874
Daily Steps,-2.414000e-01,4.537000e+00,-0.053,0.958
Systolic BP,-2.430800e+00,1.174700e+01,-0.207,0.836
Diastolic BP,2.012820e+01,1.391600e+01,1.446,0.148
Gender_Male,1.365000e+00,2.637000e+00,0.518,0.605


TODO:
    
    - maybe print some graphs before creating the model
    
    - try creating models with different subsets of variables
    
    - get predicted probabilities
    
    - calculate misclassification rates/interpret results